In [ ]:
# Example 1: build, parse and validate Azure Resource Manager resource IDs
from azure.mgmt.core.tools import (
    is_valid_resource_id, is_valid_resource_name, parse_resource_id, resource_id,
)

SUBSCRIPTION = "00000000-0000-0000-0000-000000000000"
vm_id = resource_id(subscription=SUBSCRIPTION, resource_group="example-rg",
                    namespace="Microsoft.Compute", type="virtualMachines", name="vm1")
assert vm_id == (f"/subscriptions/{SUBSCRIPTION}/resourceGroups/example-rg"
                 "/providers/Microsoft.Compute/virtualMachines/vm1"), vm_id
parts = parse_resource_id(vm_id)
assert (parts["subscription"], parts["resource_group"], parts["namespace"], parts["type"], parts["name"]) == (
    SUBSCRIPTION, "example-rg", "Microsoft.Compute", "virtualMachines", "vm1"), parts

subnet_id = resource_id(subscription=SUBSCRIPTION, resource_group="example-rg",
                        namespace="Microsoft.Network", type="virtualNetworks", name="vnet1",
                        child_type_1="subnets", child_name_1="web")
subnet = parse_resource_id(subnet_id)
assert (subnet["resource_type"], subnet["resource_name"], subnet["resource_parent"]) == (
    "subnets", "web", "virtualNetworks/vnet1/"), subnet

assert is_valid_resource_id(vm_id) and not is_valid_resource_id("vm1")
assert is_valid_resource_name("vm1") and not is_valid_resource_name("bad/name")
print("VM ID:    ", vm_id)
print("subnet ID:", subnet_id)

In [ ]:
# Example 2: ARMPipelineClient sends an authenticated request through a scripted transport
# ScriptedTransport answers each request with the next canned response, so no request leaves
# the process; FakeCredential hands out a fixed token.
import json
from azure.core.credentials import AccessToken
from azure.core.pipeline.policies import RequestIdPolicy, RetryPolicy, UserAgentPolicy
from azure.core.pipeline.transport import HttpResponse, HttpTransport
from azure.core.rest import HttpRequest
from azure.core.utils import CaseInsensitiveDict
from azure.mgmt.core import ARMPipelineClient
from azure.mgmt.core.policies import ARMChallengeAuthenticationPolicy


class StubResponse(HttpResponse):
    def __init__(self, request, status_code, payload, headers):
        super().__init__(request, None)
        self.status_code = status_code
        self.reason = "Stub"
        self.headers = CaseInsensitiveDict({"Content-Type": "application/json", **headers})
        self.content_type = "application/json"
        self._body = json.dumps(payload).encode()

    def body(self):
        return self._body


class ScriptedTransport(HttpTransport):
    def __init__(self, *responses):
        self.responses = list(responses)  # (status code, JSON payload, headers) per request
        self.requests = []

    def __enter__(self):
        return self

    def __exit__(self, *exc_info):
        pass

    def open(self):
        pass

    def close(self):
        pass

    def send(self, request, **kwargs):
        self.requests.append(request)
        status_code, payload, headers = self.responses.pop(0)
        return StubResponse(request, status_code, payload, headers)


class FakeCredential:
    def __init__(self):
        self.scopes = []

    def get_token(self, *scopes, **kwargs):
        self.scopes.append(scopes)
        return AccessToken("fake-token", 4102444800)


def arm_client(transport, credential):
    return ARMPipelineClient(
        base_url="https://management.azure.com",
        policies=[
            RequestIdPolicy(),
            UserAgentPolicy(sdk_moniker="notebook-example/1.0"),
            ARMChallengeAuthenticationPolicy(credential, "https://management.azure.com/.default"),
            RetryPolicy(retry_total=0),
        ],
        transport=transport,
    )


ARM = f"https://management.azure.com/subscriptions/{SUBSCRIPTION}"
credential = FakeCredential()
transport = ScriptedTransport((200, {"value": [{"name": "example-rg", "location": "eastus"}]}, {}))
client = arm_client(transport, credential)
response = client.send_request(HttpRequest("GET", f"{ARM}/resourcegroups?api-version=2021-04-01"))
assert response.status_code == 200
assert [group["name"] for group in json.loads(response.text())["value"]] == ["example-rg"]

(sent,) = transport.requests
assert sent.headers["Authorization"] == "Bearer fake-token", sent.headers
assert "notebook-example/1.0" in sent.headers["User-Agent"], sent.headers["User-Agent"]
assert sent.headers["x-ms-client-request-id"], "RequestIdPolicy did not add a request ID"
assert credential.scopes == [("https://management.azure.com/.default",)], credential.scopes
print(sent.method, sent.url)
print("User-Agent starts with:", sent.headers["User-Agent"].split()[0])

In [ ]:
# Example 3: an ARM error response is parsed into a structured error
from azure.core.exceptions import HttpResponseError
from azure.mgmt.core.exceptions import ARMErrorFormat

DISK = f"{ARM}/resourceGroups/example-rg/providers/Microsoft.Compute/disks/disk1?api-version=2024-03-02"
transport = ScriptedTransport(
    (404, {"error": {"code": "ResourceNotFound", "message": "The Resource 'disk1' was not found.",
                     "details": [{"code": "NotFound", "message": "disk1"}]}}, {}),
)
response = arm_client(transport, FakeCredential()).send_request(HttpRequest("GET", DISK))
error = HttpResponseError(response=response, error_format=ARMErrorFormat)
assert error.status_code == 404
assert isinstance(error.error, ARMErrorFormat), type(error.error)
assert error.error.code == "ResourceNotFound", error.error.code
assert error.error.message == "The Resource 'disk1' was not found."
assert [detail.code for detail in error.error.details] == ["NotFound"]
print(f"{error.status_code} {error.error.code}: {error.error.message}")

In [ ]:
# Example 4: ARMPolling follows the Azure-AsyncOperation header of a long-running PUT
from azure.core.polling import LROPoller
from azure.mgmt.core.polling.arm_polling import ARMPolling

operation = f"{ARM}/providers/Microsoft.Compute/locations/eastus/operations/op1?api-version=2024-03-02"
transport = ScriptedTransport(
    (201, {"name": "disk1", "properties": {"provisioningState": "Creating"}}, {"Azure-AsyncOperation": operation}),
    (200, {"status": "InProgress"}, {}),
    (200, {"status": "Succeeded"}, {}),
    (200, {"name": "disk1", "properties": {"provisioningState": "Succeeded"}}, {}),
)
client = arm_client(transport, FakeCredential())
initial = client.send_request(HttpRequest("PUT", DISK, json={"location": "eastus"}), _return_pipeline_response=True)
poller = LROPoller(client, initial, lambda pipeline_response: json.loads(pipeline_response.http_response.text()),
                   ARMPolling(timeout=0))  # timeout=0: poll again at once instead of waiting
result = poller.result()
assert poller.done() and poller.status() == "Succeeded", poller.status()
assert result == {"name": "disk1", "properties": {"provisioningState": "Succeeded"}}, result
calls = [(request.method, request.url) for request in transport.requests]
assert calls == [("PUT", DISK), ("GET", operation), ("GET", operation), ("GET", DISK)], calls
for method, url in calls:
    print(method, url)

In [ ]:
# Example 5: ARMHttpLoggingPolicy logs allow-listed headers and redacts the rest
import logging
from azure.mgmt.core.policies import ARMHttpLoggingPolicy

allowed = {name.lower() for name in ARMHttpLoggingPolicy.DEFAULT_HEADERS_ALLOWLIST}
assert {"x-ms-client-request-id", "x-ms-request-id", "x-ms-ratelimit-remaining-subscription-reads"} <= allowed
assert "authorization" not in allowed


class Collect(logging.Handler):
    def __init__(self):
        super().__init__()
        self.lines = []

    def emit(self, record):
        self.lines.append(record.getMessage())


collect = Collect()
logger = logging.getLogger("azure.core.pipeline.policies.http_logging_policy")
logger.addHandler(collect)
logger.setLevel(logging.INFO)
try:
    transport = ScriptedTransport((200, {"value": []}, {"x-ms-request-id": "request-1", "x-ms-example-header": "not-logged"}))
    client = ARMPipelineClient(
        base_url="https://management.azure.com",
        policies=[RequestIdPolicy(),
                  ARMChallengeAuthenticationPolicy(FakeCredential(), "https://management.azure.com/.default"),
                  ARMHttpLoggingPolicy(logger=logger)],
        transport=transport,
    )
    client.send_request(HttpRequest("GET", f"{ARM}/resourcegroups?api-version=2021-04-01"))
finally:
    logger.removeHandler(collect)

log = "\n".join(collect.lines)
assert "fake-token" not in log and "not-logged" not in log, log
assert "'Authorization': 'REDACTED'" in log and "'x-ms-request-id': 'request-1'" in log, log
print(log)